In [328]:
### first batch ###
# [
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch
# ]

### AA2_second_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA1_first_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA3_third_batch

# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA4_fourth_batch

In [ ]:
# {f_position}\{s_position}  AA2_second_batch\AA6_sisth_100_batch f"_pevidence.json", \AA3_third_batch\AA3_third_100_batch f"_p.json"
f_position = "AA3_third_batch"
s_position = "AA3_third_100_batch"

把文件从1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch搬到对应的D:\AA_project\AA_geo_app_regulation\dataset\1122apk\1122apk_privacy_policy_url\out_summary_json\AA2_second_batch\AA1_first_100_batch

从第一份文件里过滤，比如说从其中选择含关键词的url比如说"privacy"的url，然后去掉一些明显是广告的url

In [330]:
import json
import re
from pathlib import Path
from urllib.parse import urlparse


# -------------------------
# Privacy policy 关键词
# -------------------------

PRIVACY_KEYWORDS = [
    "privacy",
    "privacy-policy",
    "privacy_policy",
    "privacypolicy",
    "data-privacy",
    "data_privacy",
    "data-protection",
    "dataprotection",
    "personal-data",
    "gdpr",
]


# -------------------------
# 明显的广告 / 广告聚合来源
# -------------------------

AD_DOMAINS = {
    "aarki.com",
    "corp.aarki.com",

    "admixer.com",
    "admixplay.com",

    "bidmachine.io",
    "beeswax.com",

    "hybrid.ai",
    "hyperad.tech",
    "jampp.com",
    "kayzen.io",

    "persona.ly",
    "lifestreet.com",

    "discover-tech.io",
    "yeahmobi.com",
    "en.yeahmobi.com",

    "vlion.mobi",
    "mobgc.com",
    "wofhub.com",
}


def get_domain(url):
    try:
        domain = urlparse(url).netloc.lower()

        if domain.startswith("www."):
            domain = domain[4:]

        return domain

    except Exception:
        return ""


def contains_privacy_keyword(url):
    url_lower = url.lower()

    return any(
        keyword in url_lower
        for keyword in PRIVACY_KEYWORDS
    )


def is_ad_domain(url):
    domain = get_domain(url)

    for blocked_domain in AD_DOMAINS:

        if (
            domain == blocked_domain
            or domain.endswith("." + blocked_domain)
        ):
            return True

    return False


def is_from_ad_viewer(item):
    """
    如果 URL 的所有 hit 都来自类似：

    assets/ad-viewer/adViewer.xxx.js

    那么基本可以认为这是广告 SDK 内置的广告商列表，
    而不是这个 App 自己的 privacy policy。
    """

    hits = item.get("hits", [])

    if not hits:
        return False

    for hit in hits:

        file_path = (
            hit.get("file", "")
            .lower()
            .replace("\\", "/")
        )

        if "assets/ad-viewer/" not in file_path:
            return False

    return True


def evaluate_candidate(item):
    """
    返回：
        keep
        keep_reasons
        warnings
    """

    url = item.get("url", "").strip()

    keep_reasons = []
    warnings = []

    # -------------------------
    # 1. URL 必须像 privacy policy
    # -------------------------

    if not contains_privacy_keyword(url):
        return False, [], []


    keep_reasons.append(
        "privacy-related keyword in URL"
    )


    # -------------------------
    # 2. 明确广告域名
    # -------------------------

    if is_ad_domain(url):
        return False, [], []


    # -------------------------
    # 3. ad-viewer 内置广告 URL
    # -------------------------

    if is_from_ad_viewer(item):
        return False, [], []


    # -------------------------
    # 4. 一些弱信号
    # -------------------------

    domain = get_domain(url)

    third_party_domains = {
        "google.com",
        "policies.google.com",
        "firebase.google.com",

        "adjust.com",
        "appsflyer.com",
        "applovin.com",
        "vungle.com",
        "onesignal.com",
        "unity3d.com",
    }

    for third_party in third_party_domains:

        if (
            domain == third_party
            or domain.endswith("." + third_party)
        ):
            warnings.append(
                "third-party privacy policy / SDK policy"
            )
            break

    return True, keep_reasons, warnings


def parse_filename(filename):
    """
    例如：

    com.brain.game.word.quiz-23802_urls.json

    ->

    apkname:
        com.brain.game.word.quiz

    version_code:
        23802
    """

    filename = Path(filename).name

    match = re.match(
        r"(.+)-([0-9]+)_urls\.json$",
        filename
    )

    if not match:
        raise ValueError(
            f"无法从文件名解析 APK 信息: {filename}"
        )

    apkname = match.group(1)
    version_code = match.group(2)

    return apkname, version_code


def process_file(input_file, output_file):

    input_file = Path(input_file)

    apkname, version_code = parse_filename(
        input_file.name
    )

    with open(
        input_file,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)


    output_items = []

    original_urls = data.get("urls", [])


    for item in original_urls:

        keep, reasons, warnings = (
            evaluate_candidate(item)
        )

        if not keep:
            continue


        url = item.get("url", "")


        output_items.append({

            "apkname": apkname,

            "version_code": version_code,

            "url": url,

            "privacy_candidate_info": {

                "url": url,

                "keep_reasons": reasons,

                "warnings": warnings
            },

            "url_evidence_info": item
        })


    result = {
        apkname: output_items
    }


    # if output_file is None:

    #     output_file = (
    #         input_file.parent
    #         /
    #         f"{apkname}_{version_code}"
    #         f"_merged_privacy_url_evidence.json"
    #     )


    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            result,
            f,
            ensure_ascii=False,
            indent=2
        )


    print(
        f"APK: {apkname}"
    )

    print(
        f"Version: {version_code}"
    )

    print(
        f"原始 URL 数量: {len(original_urls)}"
    )

    print(
        f"Privacy candidates: {len(output_items)}"
    )

    print(
        f"输出文件: {output_file}"
    )

In [331]:
# if __name__ == "__main__":

#     process_file(
#         "com.brain.game.word.quiz-23802_urls.json"
#     )

In [332]:
source = Path(rf"1122apk\1122apk_privacy_policy_url\{f_position}\{s_position}")
target = Path(rf"1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}\{s_position}")

# source = Path(rf"1122apk\1122apk_privacy_policy_url\{f_position}")
# target = Path(rf"1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}")

# # 文件名示例: com.brain.game.word.quiz_23802_merged_privacy_url_evidence.json
# pat = re.compile(r"^(?P<apk>.+?)_(?P<num>\d+)_merged_privacy_url_evidence\.json$", re.IGNORECASE)

# # 收集所有 *_deduplicated 源目录
src_dirs = [p for p in source.glob("*") if p.is_file() and p.name.endswith("_urls.json")]
src_dirs

for src_file in src_dirs:


    apkname, version_code = parse_filename(
        src_file.name
    )

    output_file = (
        target
        /
        f"{apkname}_{version_code}"
        f"_p.json"
    )

    process_file(
        src_file, output_file
    )

    # print(f"Processing: {src_file} -> {output_file}")

APK: dinosaur.coloring.games.drawing.book
Version: 19101
原始 URL 数量: 24
Privacy candidates: 24
输出文件: 1122apk\1122apk_privacy_policy_url\out_summary_json\AA3_third_batch\AA3_third_100_batch\dinosaur.coloring.games.drawing.book_19101_p.json
APK: dinosaur.coloring.games.drawing.book
Version: 19201
原始 URL 数量: 20
Privacy candidates: 20
输出文件: 1122apk\1122apk_privacy_policy_url\out_summary_json\AA3_third_batch\AA3_third_100_batch\dinosaur.coloring.games.drawing.book_19201_p.json
APK: document.scannerapp.docscannerapp.android.imagescanner.free.camscanner.documentscanner.pdfscanner.textscanner.ocr
Version: 56
原始 URL 数量: 0
Privacy candidates: 0
输出文件: 1122apk\1122apk_privacy_policy_url\out_summary_json\AA3_third_batch\AA3_third_100_batch\document.scannerapp.docscannerapp.android.imagescanner.free.camscanner.documentscanner.pdfscanner.textscanner.ocr_56_p.json
APK: document.scannerapp.docscannerapp.android.imagescanner.free.camscanner.documentscanner.pdfscanner.textscanner.ocr
Version: 58
原始 URL 数量

指定目录1122apk\1122apk_privacy_policy_url\out_summary_json\50apk里面有很多json文档，对每个json文档得到的pp url的json文件冗余度太大。
比如说每一个json对都有apkname和version，其实一份文件apkname和version只需要出现一次即可，然后就是每一个json对一个url也只需要出现一次，不需要出现4次，有冗余。
有些json文件压根没内容，直接move到本地新的目录就行。

D:\AA_project\AA_geo_app_regulation\dataset\1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch_deduplicated结果为：同一个json对里的重复的url字段删掉就行，然后就是如果为空就不保留

加载的时候每个apk只需要加载一个即可

In [333]:
import json
import shutil
from pathlib import Path

In [334]:
### first batch ###
# [
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch
# ]

### AA2_second_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA1_first_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA7_seventh_100_batch

In [335]:
# ====== 路径配置 ======
str_p = rf"1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}\{s_position}"
# str_p = rf"1122apk\1122apk_privacy_policy_url\out_summary_json\{f_position}"
SRC_DIR = Path(rf"{str_p}")
OUT_DIR = Path(rf"{str_p}_deduplicated")
# EMPTY_DIR = Path(rf"{str_p}_empty")

OUT_DIR.mkdir(parents=True, exist_ok=True)
# EMPTY_DIR.mkdir(parents=True, exist_ok=True)
# SRC_DIR, OUT_DIR, EMPTY_DIR

In [336]:
def dedup_list_of_dict(items):
    seen = set()
    out = []
    for x in items:
        if not isinstance(x, dict):
            continue
        key = json.dumps(x, ensure_ascii=False, sort_keys=True)
        if key not in seen:
            seen.add(key)
            out.append(x)
    return out


def clean_record(rec):
    apkname = rec.get("apkname")
    version_code = rec.get("version_code")
    url = (rec.get("url") or "").strip()  # 不拆分，整串保留

    pci = rec.get("privacy_candidate_info") or {}
    uei = rec.get("url_evidence_info") or {}

    keep_reasons = pci.get("keep_reasons", [])
    warnings = pci.get("warnings", [])

    raw_hits = uei.get("hits", []) if isinstance(uei.get("hits", []), list) else []
    hits = []
    for h in raw_hits:
        if isinstance(h, dict):
            h2 = dict(h)
            h2.pop("url", None)  # 去掉内部重复 url
            hits.append(h2)
    hits = dedup_list_of_dict(hits)

    raw_usage = uei.get("usage", []) if isinstance(uei.get("usage", []), list) else []
    usage = dedup_list_of_dict([u for u in raw_usage if isinstance(u, dict)])

    out = {
        "apkname": apkname,
        "version_code": version_code,
        "url": url
    }

    # 只有非空才保留 privacy_candidate_info
    pci_out = {}
    if isinstance(keep_reasons, list) and keep_reasons:
        pci_out["keep_reasons"] = keep_reasons
    if isinstance(warnings, list) and warnings:
        pci_out["warnings"] = warnings
    if pci_out:
        out["privacy_candidate_info"] = pci_out

    # usage 为空不保留；hits 为空也不保留
    uei_out = {}
    if hits:
        uei_out["hits"] = hits
    if usage:
        uei_out["usage"] = usage
    if uei_out:
        out["url_evidence_info"] = uei_out

    return out


def merge_record(base, new):
    """同一顶层url的记录合并"""
    # 合并 keep_reasons / warnings
    br = base["privacy_candidate_info"].get("keep_reasons", [])
    nr = new["privacy_candidate_info"].get("keep_reasons", [])
    bw = base["privacy_candidate_info"].get("warnings", [])
    nw = new["privacy_candidate_info"].get("warnings", [])

    base["privacy_candidate_info"]["keep_reasons"] = list(dict.fromkeys(br + nr))
    base["privacy_candidate_info"]["warnings"] = list(dict.fromkeys(bw + nw))

    # 合并 hits / usage 并去重
    base_hits = base["url_evidence_info"].get("hits", [])
    new_hits = new["url_evidence_info"].get("hits", [])
    base_usage = base["url_evidence_info"].get("usage", [])
    new_usage = new["url_evidence_info"].get("usage", [])

    base["url_evidence_info"]["hits"] = dedup_list_of_dict(base_hits + new_hits)
    base["url_evidence_info"]["usage"] = dedup_list_of_dict(base_usage + new_usage)

    return base


summary = {"total": 0, "processed": 0, "moved_empty": 0, "failed": 0}

for fp in SRC_DIR.glob("*.json"):
    summary["total"] += 1
    try:
        txt = fp.read_text(encoding="utf-8", errors="ignore").strip()
        # if not txt:
        #     shutil.move(str(fp), str(EMPTY_DIR / fp.name))
        #     summary["moved_empty"] += 1
        #     continue

        try:
            obj = json.loads(txt)
        except json.JSONDecodeError:
            # shutil.move(str(fp), str(EMPTY_DIR / fp.name))
            # summary["moved_empty"] += 1
            continue

        # 兼容你的结构：{apk_id: [records]}
        if isinstance(obj, dict) and len(obj) == 1 and isinstance(next(iter(obj.values())), list):
            apk_id = next(iter(obj.keys()))
            records = next(iter(obj.values()))
        else:
            # 兜底
            apk_id = fp.stem
            if isinstance(obj, list):
                records = obj
            elif isinstance(obj, dict):
                records = [obj]
            else:
                records = []

        cleaned_map = {}  # key=顶层url(整串，不拆分)

        for rec in records:
            if not isinstance(rec, dict):
                continue
            cleaned = clean_record(rec)
            url_key = cleaned["url"]
            if not url_key:
                continue

            if url_key not in cleaned_map:
                cleaned_map[url_key] = cleaned
            else:
                cleaned_map[url_key] = merge_record(cleaned_map[url_key], cleaned)

        out_records = list(cleaned_map.values())

        if not out_records:
            # shutil.move(str(fp), str(EMPTY_DIR / fp.name))
            # summary["moved_empty"] += 1
            continue

        out_obj = {apk_id: out_records}
        out_fp = OUT_DIR / fp.name
        out_fp.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2), encoding="utf-8")

        summary["processed"] += 1

    except Exception as e:
        summary["failed"] += 1
        print(f"[FAILED] {fp.name}: {e}")

print("处理完成：", summary)
print("去重输出目录:", OUT_DIR)
# print("空文件移动目录:", EMPTY_DIR)

处理完成： {'total': 104, 'processed': 98, 'moved_empty': 0, 'failed': 0}
去重输出目录: 1122apk\1122apk_privacy_policy_url\out_summary_json\AA3_third_batch\AA3_third_100_batch_deduplicated


In [337]:
### first batch ###
# [
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch\AA6_sisth_74_batch
# ]

<!-- 指定目录为1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch，遍历所有以'_deduplicated'结尾的文件，然后遍历里面的文档，文件名是com.brain.game.word.quiz_23802_merged_privacy_url_evidence，其中com.brain.game.word.quiz是apkname，23802是编号，之后只根据文档名，一个apk只保存一个json文档，虽然编号不一样，但是内容是一样的，所以一个apk只需要保存一个json文档。将结果输出到新的目录吧。 -->

In [338]:
# import re
# import json
# import hashlib
# import shutil
# from pathlib import Path

In [339]:
# ROOT = Path(r"1122apk\1122apk_privacy_policy_url\out_summary_json\AA1_first_batch")

# # 文件名示例: com.brain.game.word.quiz_23802_merged_privacy_url_evidence.json
# pat = re.compile(r"^(?P<apk>.+?)_(?P<num>\d+)_merged_privacy_url_evidence\.json$", re.IGNORECASE)

# # 收集所有 *_deduplicated 源目录
# src_dirs = [p for p in ROOT.iterdir() if p.is_dir() and p.name.endswith("_deduplicated")]

In [340]:
# global_total = 0
# global_copied = 0
# global_dup = 0
# global_bad = 0

# for src_dir in sorted(src_dirs):
#     # 每个源目录对应一个新的输出目录
#     out_dir = ROOT / f"{src_dir.name}_apk_unique"
#     out_dir.mkdir(parents=True, exist_ok=True)

#     seen_apk = set()
#     total = copied = dup = bad = 0

#     for fp in sorted(src_dir.glob("*.json")):
#         total += 1
#         m = pat.match(fp.name)
#         if not m:
#             bad += 1
#             continue

#         apk = m.group("apk")
#         if apk in seen_apk:
#             dup += 1
#             continue

#         seen_apk.add(apk)

#         # 每个 apk 只保留一个文件（文件名去掉编号）
#         out_name = f"{apk}_merged_privacy_url_evidence.json"
#         shutil.copy2(fp, out_dir / out_name)
#         copied += 1

#     global_total += total
#     global_copied += copied
#     global_dup += dup
#     global_bad += bad

#     print(f"[{src_dir.name}]")
#     print(f"  扫描: {total}")
#     print(f"  输出: {copied}")
#     print(f"  跳过重复apk: {dup}")
#     print(f"  跳过文件名不匹配: {bad}")
#     print(f"  输出目录: {out_dir}")

# print("\n=== 全部完成 ===")
# print("总扫描:", global_total)
# print("总输出:", global_copied)
# print("总跳过重复apk:", global_dup)
# print("总跳过文件名不匹配:", global_bad)

将clause里的json段正常加载然后保存为json。

In [341]:
# import json
# from pathlib import Path

In [342]:
# raw_items = [
# {"P1": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to carefully review the segment to determine whether it specifies the categories or specific pieces of personal information that the business collects or uses. \n\nFocus: \nAnalyze exclusively for personal information being collected or used by the business controller or processor. Ignore personal information related solely to third-party data sharing, disclosure, or selling. \n\nDefinition of Personal Information: \nPersonal Information includes, but is not limited to: financial, commercial, health, contact, geolocation, demographic, biometric, personal identifier, user profile, social media data, IP address and device IDs, tracking elements (Cookies, beacon, etc.), computer information, Internet or other electronic network activity information, survey data, and other sensitive personal information under the CCPA. \nGeneral mentions of personal information (e.g., certain personal information) without specifying categories, or details do not qualify as valid.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- presence_flag: Set to '1' if the segment explicitly specifies the categories or specific pieces of personal information that the business controller collects or uses, or '0' if it does not.\n- extracted_text: A list containing the exact text of each identified category or specific piece of personal information that the business controller collects or uses. If none is present, return 'None'."},
# {"P2": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to carefully review the segment to determine whether it specifies the categories or specific pieces of personal information that the business controller shares with third parties (including the public), or that may be used or collected by third-party service providers. \n\nFocus: \nAnalyze exclusively for personal information being shared to third-parties, or may be used or collected by third-parties. Ignore personal information related solely to first-party data collection or processing.\n\nCategories of Personal Information: \nPersonal Information includes, but is not limited to: financial, commercial, health, contact, geolocation, demographic, biometric, personal identifier, user profile, social media data, IP address and device IDs, tracking elements (Cookies, beacon, etc.), computer information, Internet or other electronic network activity information, survey data, and other sensitive personal information under CCPA. \nGeneral mentions of personal information (e.g., certain personal information) without specifying categories, or details do not qualify as valid.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- presence_flag: Set to '1' if the segment specifies any categories or specific pieces of personal information being shared with or used by third-parties, or '0' if it does not.\n- extracted_text: A list containing the exact text of each identified category or specific piece of personal information being shared with or used by third-parties. If none is present, return 'None'."},
# {"P3": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to carefully review the segment to determine whether it specifies the categories or specific pieces of personal information that the business controller disclose and sell behaviors with third parties (including the public), or that may be used or collected by third-party service providers. \n\nFocus: \nAnalyze exclusively for personal information being disclosed/sold to third-parties, or may be used or collected by third-parties. Ignore personal information related solely to first-party data collection or processing.\n\nCategories of Personal Information: \nPersonal Information includes, but is not limited to: financial, commercial, health, contact, geolocation, demographic, biometric, personal identifier, user profile, social media data, IP address and device IDs, tracking elements (Cookies, beacon, etc.), computer information, Internet or other electronic network activity information, survey data, and other sensitive personal information under CCPA. \nGeneral mentions of personal information (e.g., certain personal information) without specifying categories, or details do not qualify as valid.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- presence_flag: Set to '1' if the segment specifies any categories or specific pieces of personal information being shared with or used by third-parties, or '0' if it does not.\n- extracted_text: A list containing the exact text of each identified category or specific piece of personal information being shared with or used by third-parties. If none is present, return 'None'."},
# {"P4": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether the segment includes the information about the purposes for the business to collect or use consumers' personal information. \n\nFocus: \nAnalyze exclusively for purpose information related to first-party collection/use of data. Ignore any purpose information related solely to third-party data sharing, disclosure, or selling. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- presence_flag: Set to '1' if the segment includes the purpose information for first-party data collection/use, or '0' if it does not. \n- extracted_text: A list containing the exact text of each purpose information for first-party data collection/use identified in the segment. If none is present, return 'None'."},
# {"P5": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether the segment includes information about the purposes for the business to share consumers' personal information to third parties (including the public). \n\nFocus: \nAnalyze exclusively for purpose information related to third-party data sharing. If the purpose can be inferred from the naming of a third party (e.g., \"third-party advertising service\"), it also qualifies. Ignore any purpose information related solely to first-party data collection or use.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- presence_flag: Set to '1' if the segment includes the purpose information for third-party data sharing, disclosure, or selling, or '0' if it does not. \n- extracted_texts: A list containing the exact text of each purpose information for third-party data sharing, disclosure or selling identified in the segment. If none is present, return 'None'."},
# {"P6": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether the segment includes information about the purposes for the business to disclosure and selling behaviors consumers' personal information to third parties (including the public). \n\nFocus: \nAnalyze exclusively for purpose information related to third-party data disclosing and selling. If the purpose can be inferred from the naming of a third party (e.g., \"third-party advertising service\"), it also qualifies. Ignore any purpose information related solely to first-party data collection or use.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- presence_flag: Set to '1' if the segment includes the purpose information for third-party data sharing, disclosure, or selling, or '0' if it does not. \n- extracted_texts: A list containing the exact text of each purpose information for third-party data sharing, disclosure or selling identified in the segment. If none is present, return 'None'."},
# {"P7": "\n\nTask Description and Instructions:\nI will provide a segment of a business's privacy policy from their website in Markdown format\nYour task is to review the segment to determine whether it explicitly specifies the categories or specific items of Personal Data collected by the business AND whether the segment clearly indicates which items are Mandatory to collect and which items are Optional to collect. \n\nMandatory vs Optional definitions:\n- Mandatory: The segment clearly indicates the Personal Data is required to provide/use the service or is automatically collected by the system (e.g., \"we collect automatically,\" \"required,\" \"must provide\").\n- Optional: The segment clearly indicates the Personal Data is collected only if the user chooses to provide it or only in certain situations (e.g., \"if you choose to provide,\" \"you may provide,\" \"optional\"). \n\nValidity rules:\n- The segment qualifies ONLY if it lists at least one specific category or specific item of Personal Data AND assigns each listed item to either Mandatory or Optional.\n- Generic mentions of \"personal information\" or \"personal data\" without listing specific categories/items do NOT qualify.\n- If the segment lists categories/items but does NOT clearly indicate Mandatory vs Optional for each item, it does NOT qualify. \n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_items> ].\n- presence_flag: Set to '1' if the segment qualifies under the validity rules above; otherwise set to '0'.\n- extracted_items: If presence_flag is '1', return a list of objects where each object includes (a) the exact text of the Personal Data category/item and (b) whether it is \"Mandatory\" or \"Optional\". If presence_flag is '0', return 'None'."},
# {"P8": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains any information about the retention period for which consumers\u2019 personal data will be stored. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- presence_flag: Set to '1' if the segment includes relevant retention information, or '0' if it does not. \n- extracted_text: A list containing the exact text of each retention information identified in the segment. If none is present, return 'None'."},
# {"P10": "\n\nTask Description and Instructions:\nI will provide a segment of a business's privacy policy from their website in Markdown format.\nYour task is to review the segment carefully to determine whether it explicitly describes the methods by which the business collects personal information.\n\nCollection methods include, for example:\n- Information provided directly by the user (e.g., account registration forms, checkout, surveys, customer support communications).\n- Information collected automatically (e.g., cookies, pixels, log files, device identifiers, IP address, analytics technologies).\n- Information collected from or through third parties (e.g., third-party SDKs, advertising partners, analytics providers) ONLY if described as a collection method.\n\nValidity rules:\n- Generic statements such as \"we collect personal information\" without describing HOW it is collected do NOT qualify.\n- The segment qualifies only if it states a concrete collection method/channel/technology (e.g., \"through forms,\" \"automatically,\" \"via cookies,\" \"from your device,\" \"from third-party services\").\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment includes one or more personal information collection methods; otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each collection method statement identified in the segment. Exclude unrelated content. If none is present, return 'None'."},
# {"P11": "\n\nTask Description and Instructions:\nI will provide a segment of a business's privacy policy from their website in Markdown format.\nYour task is to review the segment carefully to determine whether it explicitly describes the methods by which the business processes personal information.\n\nProcessing methods include, for example:\n- Storing, retaining, archiving, or backing up personal information (e.g., \"store\", \"retain\", \"archive\", \"backup\").\n- Securing personal information through technical or organizational measures (e.g., \"encrypt\", \"pseudonymize\", \"anonymize\", \"access controls\", \"authentication\").\n- Using automated or manual techniques to analyze or handle data (e.g., \"analytics\", \"profiling\", \"automated processing\", \"manual review\").\n- Combining, matching, or enriching data from different sources (e.g., \"combine\", \"link\", \"match\", \"merge\").\n- Transmitting or transferring data within systems for operational purposes (e.g., \"transfer\", \"sync\", \"process on servers\").\n\nValidity rules:\n- Generic statements such as \"we process personal information\" or \"we use personal information\" without describing HOW it is processed do NOT qualify.\n- The segment qualifies only if it states at least one concrete processing operation or technique (e.g., encryption, storage, retention, anonymization, profiling, automated decision-making, access control, backups, data matching/merging).\n- Do not treat collection methods (e.g., \"through forms\", \"via cookies\") as processing methods unless the text also describes a processing operation performed on the collected data.\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment includes one or more personal information processing methods; otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each processing method statement identified in the segment. Exclude unrelated content. If none is present, return 'None'."},
# {"P12": "\n\nTask Description and Instructions:\nI will provide a segment of a business's privacy policy from their website in Markdown format.\nYour task is to carefully review the segment to determine whether it explicitly describes the methods by which the business stores personal data (i.e., where and how personal data is stored and maintained).\n\nStorage methods include, for example:\n- Storage location or environment (e.g., \"on our servers\", \"in the cloud\", \"data centers\", \"on your device\", \"local storage\").\n- Storage format or medium (e.g., \"databases\", \"log files\", \"encrypted storage\", \"backups\", \"archives\").\n- Storage-related security applied to stored data (e.g., \"encryption at rest\", \"access controls\", \"restricted access\").\n- Backup and disaster recovery practices (e.g., \"backups\", \"redundant storage\", \"disaster recovery\").\n\nValidity rules:\n- Generic statements such as \"we store your personal data\" without describing WHERE or HOW it is stored do NOT qualify.\n- The segment qualifies only if it provides at least one concrete storage method detail (e.g., storage location, medium/format, storage security such as encryption at rest, backup/archiving approach).\n- Do not treat retention periods (how long data is kept) as storage methods unless the text also describes the storage mechanism or location.\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment includes one or more personal data storage methods; otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each storage method statement identified in the segment. Exclude unrelated content. If none is present, return 'None'."},
# {"P13": "\n\nTask Description and Instructions:\nI will provide a segment of a business's privacy policy from their website in Markdown format.\nYour task is to carefully review the segment to determine whether it explicitly describes the methods by which the business destroys, deletes, disposes of, or irreversibly de-identifies personal data.\n\nDestruction methods include, for example:\n- Deletion or erasure methods (e.g., \"delete\", \"erase\", \"wipe\", \"permanently delete\", \"secure deletion\").\n- Physical disposal methods for storage media (e.g., \"shred\", \"destroy\", \"dispose of\", \"physically destroy media\").\n- Irreversible de-identification methods used instead of deletion (e.g., \"anonymize\", \"irreversibly de-identify\", \"aggregate so it can no longer identify you\").\n- Destruction-related procedures (e.g., \"secure disposal\", \"data sanitization\", \"overwrite\", \"cryptographic erasure\").\n\nValidity rules:\n- Generic statements such as \"we delete your personal data\" or \"we will remove your information\" without describing HOW it is destroyed/deleted do NOT qualify.\n- The segment qualifies only if it describes at least one concrete destruction method or technique (e.g., secure deletion, wiping/overwriting, shredding physical media, cryptographic erasure, irreversible anonymization).\n- Statements that only describe WHEN data is deleted (retention periods) without describing HOW it is deleted do NOT qualify.\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment includes one or more personal data destruction methods; otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each destruction method statement identified in the segment. Exclude unrelated content. If none is present, return 'None'."},
# {"P14": "\n\nTask Description and Instructions:\nI will provide a segment of a business's privacy policy from their website in Markdown format.\nYour task is to review the segment carefully to determine whether it states that an individual has the right to request information from the data controller about (1) the purpose(s) for processing their personal data, and (2) whether their personal data is used/processed in compliance with those stated purposes.\n\nFocus:\nAnalyze only for statements describing the individual’s right to:\n- learn or be informed of the purpose(s) of processing their personal data; and/or\n- learn or confirm whether their personal data is used/processed in accordance with those purposes.\n\nValidity rules:\n- The segment qualifies if it explicitly mentions the right to know/learn the purpose(s) of processing and/or whether processing/usage is consistent with the stated purpose(s).\n- Generic references to \"your rights\" without mentioning purpose(s) of processing or compliance with purpose(s) do NOT qualify.\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment contains one or more statements granting the above right(s); otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each relevant statement identified in the segment. If none is present, return 'None'."},
# {"P15": "\n\nTask Description and Instructions:\nI will provide a segment of a business's privacy policy from their website in Markdown format.\nYour task is to review the segment carefully to determine whether it informs the Data Subject, at the time of collection (or upon collection), of:\n(1) the entities to which Personal Data will be disclosed,\n(2) the capacity/role of such entities (e.g., service provider, processor, affiliate, third party), and\n(3) whether Personal Data will be transferred, disclosed, or otherwise processed outside the Kingdom of Saudi Arabia.\n\nFocus:\nAnalyze only for notice content addressing (a) disclosure recipients and their capacity/role, and (b) cross-border transfer/processing outside the Kingdom.\n\nValidity rules:\n- The segment qualifies if it explicitly identifies the disclosure recipients (or recipient categories) AND describes their capacity/role, and/or explicitly states whether Personal Data will be transferred/disclosed/processed outside the Kingdom.\n- Generic statements such as \"we may share data with third parties\" without identifying recipient entities/categories or their capacity/role do NOT qualify for the recipient/capacity requirement.\n- Statements about international transfer/processing qualify only if they explicitly reference transfer/processing outside the Kingdom (e.g., \"outside Saudi Arabia\" / \"outside the Kingdom\").\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment contains one or more statements addressing the above notice elements; otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each relevant statement identified in the segment. If none is present, return 'None'."},
# {"P16": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains the information about the fact that the controller intends to transfer personal data to a third country or international organisation and the existence or absence of an adequacy decision by the Commission, or references to suitable safeguards and how to access them.\n\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"P17": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it includes specific notices regarding the sale of sensitive personal data. Follow the instructions below carefully and respond in the specified JSON format. \nSteps to Perform: \n1.  Check for Notice 1: Determine whether the segment contains the following notice: \n\"NOTICE: This website may sell your sensitive personal data.\"\n2.  Check for Notice 2: Determine whether the segment contains the following notice: \n\"NOTICE: We may sell your sensitive personal data.\"\n\nResponse Format:\nPlease respond in the following JSON format: \n[ \n    <presence_flag_1>, <extracted_text_1>, \n    <presence_flag_2>, <extracted_text_2>\n]\n- presence_flag_1: Set to '1' if the segment includes Notice 1, or '0' if it does not.\n- extracted_text_1: Return the exact text of Notice 1 identified in the segment. If none is present, return 'None'.\n- presence_flag_2: Set to '1' if the segment includes Notice 2, or '0' if it does not. \n- extracted_text_2: Return the exact text of Notice 2 identified in the segment. If none is present, return 'None'."},
# {"P18": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it includes specific notices regarding the sale of biometric personal data. Follow the instructions below carefully and respond in the specified JSON format. \nSteps to Perform: \n1.  Check for Notice 1: Determine whether the segment contains the following notice: \n\"NOTICE: This website may sell your biometric personal data.\"\n2.  Check for Notice 2: Determine whether the segment contains the following notice: \n\"NOTICE: We may sell your biometric personal data.\"\n\nResponse Format:\nPlease respond in the following JSON format: \n[ \n    <presence_flag_1>, <extracted_text_1>, \n    <presence_flag_2>, <extracted_text_2>\n]\n- presence_flag_1: Set to '1' if the segment includes Notice 1, or '0' if it does not.\n- extracted_text_1: Return the exact text of Notice 1 identified in the segment. If none is present, return 'None'.\n- presence_flag_2: Set to '1' if the segment includes Notice 2, or '0' if it does not. \n- extracted_text_2: Return the exact text of Notice 2 identified in the segment. If none is present, return 'None'."},
# {"P19": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it provides a statement regarding whether the business uses or discloses sensitive personal information for purposes other than those specified in section 7027, subsection (m) of the California Consumer Privacy Act (CCPA). \n\nSummary of permissible purposes specified in Section 7027, Subsection (m):\n- Providing expected goods or services.\n- Addressing security, fraud, or illegal activities.\n- Ensuring safety and performing business services.\n- Short-term, transient use (e.g., ads) without profiling or third-party disclosure.\n- Improving or verifying products/services.\n- Collecting/processing sensitive data without inferring consumer characteristics\n\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes the relevant statement, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each statement identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"P20": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it includes statements related to processing children\u2019s personal information.\nSteps to Perform:\n1. Check for Statement 1: Whether the segment contains a statement regarding whether the business has actual knowledge that it sells or shares the personal information of consumers under 16 years of age, as required under the California Consumer Privacy Act (CCPA).\n2. Check for Statement 2: Whether the segment contains a statement regarding whether the business has actual knowledge of collecting personal information online from a child under 13 years of age, as required under the Children\u2019s Online Privacy Protection Act (COPPA). \n\nResponse Format: \nRespond using the following JSON format: \n[ \n    <presence_flag_1>, <extracted_text_1>, \n    <presence_flag_2>, <extracted_text_2>\n]\n- <presence_flag_1>: Set to '1' if the segment includes Statement 1, or  '0' if it does not.\n- <extracted_text_1>: Return the exact text of Statement 1 as identified in the segment. If none is present, return 'None'.\n- <presence_flag_2>: Set to '1' if the segment includes Statement 2, or  '0' if it does not.\n- <extracted_text_2>: Return the exact text of Statement 2 as identified in the segment.  If none is present, return 'None'.\nDo not include any additional explanations outside the JSON format to ensure ease of processing."},
# {"P21": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it provides information on whether the provision of personal data is a statutory or contractual requirement, or a requirement necessary to enter into a contract, as well as whether the data subject is obliged to provide the personal data and of the possible consequences of failure to provide such data, as required under the GDPR.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes the relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"CR1": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app’s privacy policy in Markdown format.\nYour task is to review the segment carefully to determine whether it identifies:\n(1) the data controller (i.e., the entity that determines the purposes and means of processing personal data), and\n(2) the controller’s representative, if any (e.g., an authorized/local representative in a specific country/region).\n\nFocus:\nAnalyze only for statements that explicitly name the controller and/or explicitly identify a representative. This requirement is about identifying WHO is responsible for the app’s data processing, not about the categories of data collected.\n\nValidity rules:\n- Controller identity qualifies only if the segment clearly names the controller (e.g., company/legal entity name, developer name, operator name) or clearly states “the controller is …”.\n- Generic labels such as “we”, “our app”, or “the developer” without naming an entity do NOT qualify.\n- Contact information (email/address/phone) qualifies only if it is explicitly associated with the controller identity (e.g., “Controller: X” followed by contact details).\n- Representative information qualifies only if the segment explicitly indicates a representative exists (e.g., “our representative”, “authorized representative”, “local representative”) and provides identifying details (name/entity and/or clear representative designation and contact details).\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment identifies the data controller and/or its representative (if any); otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each statement that identifies the controller and/or the representative. If none is present, return 'None'."},
# {"CR2 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Please review the segment carefully to determine whether it contains contact details for the business controller or the data protection officer. Contact details can be either online (e.g., an email address) or offline (e.g., a physical address). \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes contact details, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each contact information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"CR3": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app’s privacy policy in Markdown format.\nYour task is to review the segment carefully to determine whether it states the address/residence/place of establishment of the controller’s representative (if a representative is designated/required).\n\nFocus:\nAnalyze only for statements that provide a physical location for the representative, such as a mailing address, registered office address, place of residence, or place of establishment (e.g., street address, city/country, registered office).\nDo NOT treat a general controller address as the representative’s address unless the text explicitly identifies it as the representative’s address.\n\nValidity rules:\n- The segment qualifies only if it explicitly indicates a representative exists (e.g., \"representative\", \"authorized representative\", \"local representative\") AND provides the representative’s address/residence/place of establishment.\n- Email addresses, phone numbers, or web forms alone do NOT qualify as an address/residence/place.\n- A generic statement like \"contact our representative\" without a physical location does NOT qualify.\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment provides the representative’s address/residence/place (as defined above); otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each statement that includes the representative’s address/residence/place. If none is present, return 'None'."},
# {"CR4": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to carefully review the segment to determine whether it specifies the categories or specific sources from which the personal information is collected by the business.\n\nFocus:\nAnalyze exclusively for the sources from which the business collects personal information. Exclude any recipients with whom the business shares data. \n\nCategories of sources: \n\"Categories of sources\" means types or groupings of persons or entities from which a business collects personal information about consumers, described with enough particularity to provide consumers with a meaningful understanding of the type of person or entity. They may include the consumer directly, advertising networks, internet service providers, data analytics providers, government entities, operating systems and platforms, social networks, and data brokers.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- presence_flag: Set to '1' if the segment specifies the categories or specific sources from which the personal information is collected, or '0' if it does not.\n- extracted_text: A list containing the exact text of each identified category or specific source from which the personal information is collected. If none is present, return 'None'."},
# {"CR6": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app’s privacy policy in Markdown format.\nYour task is to review the segment carefully to determine whether it specifies the categories of recipients to whom personal data may be shared/disclosed/transferred.\n\nFocus:\nAnalyze only for recipient categories (i.e., who receives the data), such as \"service providers\", \"analytics providers\", \"advertising partners\", \"payment processors\", \"cloud/hosting providers\", \"affiliates\", \"business partners\", \"authorities/law enforcement\", etc.\nDo NOT analyze for the categories of personal data shared (what data), unless it is necessary to identify the recipient category.\n\nValidity rules:\n- The segment qualifies if it explicitly lists one or more categories of recipients.\n- Generic statements like \"we may share your data with third parties\" without describing recipient categories do NOT qualify.\n- Naming specific third parties (e.g., Google, Firebase) qualifies and should be treated as identifying recipients (and you may also infer the recipient category only if the text explicitly states it, e.g., \"analytics provider\").\n\nResponse Format:\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ].\n- presence_flag: Set to '1' if the segment specifies one or more categories of data-sharing recipients; otherwise set to '0'.\n- extracted_texts: A list containing the exact text of each statement that identifies the recipient category (or names specific recipients). If none is present, return 'None'."},
# {"R1": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to appeal. The presence of a relevant heading text (e.g., \"Right to Appeal \") also qualifies.  \nNote: This task is distinct from determining whether the consumer has the right to lodge a complaint. Solely mentioning the right to lodge a complaint does not qualify as information on how to appeal a controller's decision. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R2": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app's privacy policy in Markdown format. Please review the segment carefully to determine whether it includes users' right to request the data controller (or equivalent terms such as \"we\", \"company\", \"data controller\") to confirm whether their personal data is being processed (i.e., the right to know whether processing occurs).\n\nQualification Criteria (any of the following qualifies):\n1) Explicit statement that a user/data subject may ask/request confirmation of whether their personal data is processed (e.g., \"confirm whether we process your personal data\").\n2) Right of access language that clearly includes confirmation of processing (e.g., \"you have the right to access your personal data and obtain confirmation whether it is being processed\").\n3) A clearly relevant heading indicating this right (e.g., \"Right to Know Whether Data Is Processed\", \"Right to Confirmation of Processing\", \"Right of Access\") AND the surrounding text supports that meaning.\n\nNon-qualifying examples (do NOT qualify):\n- General statements saying the company \"processes\" or \"uses\" data, without granting users a right to request/learn whether processing occurs.\n- Purely describing categories of data collected/processed, without a user request right.\n- Generic \"contact us\" language that does not mention confirming whether data is processed.\n\nResponse Format:\nRespond using the following JSON format: [<presence_flag>, <extracted_texts>].\n- <presence_flag>: Set to '1' if the segment includes qualifying information, or '0' if it does not.\n- <extracted_texts>: A list containing the exact text of each qualifying piece of information identified (include relevant headings if they qualify). Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R4": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app's privacy policy in Markdown format. Please review the segment carefully to determine whether it states that an individual (user/data subject) has the right to request the data controller (or equivalent terms such as \"we\", \"company\") and (a) learn the purpose(s) of processing of their personal data, and/or (b) learn whether their personal data are used/processed in compliance with those stated purposes.\n\nQualification Criteria (any of the following qualifies):\n1) Explicit right for the user to request/obtain information about the purpose(s) for which their personal data are processed.\n2) Explicit right for the user to request/obtain information on whether their personal data are used/processed in accordance with the declared purpose(s) (purpose limitation / compliance with purpose).\n3) Right of access wording that clearly includes purpose-of-processing information and/or whether processing is consistent with the purpose, even if bundled with other access rights.\n4) A clearly relevant heading (e.g., \"Right of Access\", \"Your Rights\", \"Purposes of Processing\", \"Purpose Limitation\") ONLY if the surrounding text grants the user a right to request/learn the above (a) and/or (b).\n\nImportant Notes:\n- Either (a) purpose-of-processing OR (b) compliance-with-purpose is sufficient to qualify, but extract only the exact text that matches.\n- General statements listing purposes (e.g., \"we use your data to...\") do NOT qualify unless the policy frames them as a user right to request/learn.\n\nNon-qualifying examples (do NOT qualify):\n- Purely descriptive purpose lists without a user request/right (\"we process data for...\").\n- Generic transparency statements (\"we are committed to using data appropriately\") without granting a user right to request/learn purposes or compliance.\n- Contact-us language that does not mention requesting/obtaining purpose or purpose-compliance information.\n\nResponse Format:\nRespond using the following JSON format: [<presence_flag>, <extracted_texts>].\n- <presence_flag>: Set to '1' if the segment includes qualifying information, or '0' if it does not.\n- <extracted_texts>: A list containing the exact text of each qualifying piece of information identified (include relevant headings if they qualify). Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R5": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app's privacy policy in Markdown format. Please review the segment carefully to determine whether it states that an individual (user/data subject) has the right to request the data controller (or equivalent terms such as \"we\", \"company\") and to know which third parties their personal data are transferred/shared/disclosed to, whether within the country or abroad (cross-border).\n\nQualification Criteria (any of the following qualifies):\n1) Explicit right for the user to request/obtain a list or categories of third parties (recipients) to whom their personal data are transferred/shared/disclosed.\n2) Explicit right for the user to request/obtain information about cross-border/international transfers AND the recipients/third parties involved (e.g., \"third parties abroad\" / \"recipients in other countries\").\n3) Right of access wording that clearly includes \"recipients\" / \"third parties\" / \"to whom data is disclosed\" (even if bundled with other access rights), including domestic and/or international recipients.\n4) A clearly relevant heading (e.g., \"Right to Know Recipients\", \"Third-Party Recipients\", \"Data Transfers\", \"International Transfers\", \"Right of Access\") ONLY if the surrounding text grants the user a right to request/know recipients.\n\nImportant Notes:\n- The key requirement is a USER RIGHT to know/request recipients. Merely listing third parties or stating that data may be shared does NOT qualify unless framed as a right to request/know.\n- If the text mentions \"transferred\" without identifying recipients but says the user can learn \"to whom\" it is transferred, that qualifies.\n\nNon-qualifying examples (do NOT qualify):\n- Descriptions that the company shares/transfers data with third parties, without granting a right to request/know who they are.\n- Generic statements like \"we may share with partners/service providers\" without a request/right framing.\n- Contact-us language without mentioning requesting recipient/third-party transfer information.\n\nResponse Format:\nRespond using the following JSON format: [<presence_flag>, <extracted_texts>].\n- <presence_flag>: Set to '1' if the segment includes qualifying information, or '0' if it does not.\n- <extracted_texts>: A list containing the exact text of each qualifying piece of information identified (include relevant headings if they qualify). Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R6": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Please review the segment carefully to determine whether it contains information about customers' right to confirm or consent for the processing of their personal data. The presence of a relevant heading text (e.g., \"Right to Consent\", \"Right to Comfirm\") also qualifies. Statements merely mentioning customers providing consent or confirming do not qualify. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R7": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to know, access or confirm what personal information of the consumer the business has processed or is processing. The presence of a relevant heading text (e.g., \"Right to Access\") also qualifies. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R8": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to delete or erase personal data provided by or obtained about the consumer. The presence of a relevant heading text (e.g., \"Right to Delete\") also qualifies. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R9": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to correct or rectify inaccuracies in the consumer's personal data. The presence of a relevant heading text (e.g., \"Right to Correct\") also qualifies. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R10": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to object to the processing of personal data concerning them. The presence of a relevant heading text (e.g., \"Right to Object\") also qualifies. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R11": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to limit or restrict the use or disclosure of the sensitive personal information by the business. The presence of a relevant heading text (e.g., \"Right to Limit\") also qualifies. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R13 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Please review the segment carefully to determine whether it contains information about customers' right to withdraw consent for the processing of their personal data. The presence of a relevant heading text (e.g., \"Right to Withdraw Consent\") also qualifies. Statements merely mentioning customers providing or withholding consent do not qualify.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R14 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to data portability. This right enables consumers to obtain a copy of their personal data in a portable and, to the extent technically feasible, readily usable format that allows the consumer to transmit the data to another entity. Note that the presence of a simple statement indicating that consumers can request a copy of their personal data qualifies. The presence of a relevant heading text (e.g., \"Right to Data Portability\") also qualifies. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R15 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to lodge a complaint. The presence of a relevant heading text (e.g., \"Right to Lodge a Complaint\") also qualifies. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R16": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app's privacy policy in Markdown format. Please review the segment carefully to determine whether it includes the consumer's right to know what personal information is sold or shared and to whom (and/or disclosed for a business purpose), consistent with CCPA/CPRA-style rights.\n\nQualification Criteria (any of the following qualifies):\n1) The policy states that consumers/users can submit a request (e.g., \"Right to Know\", \"Access Request\", \"Verifiable Consumer Request\") to learn categories of personal information collected about them.\n2) The policy states that consumers/users can request disclosure of categories of personal information sold or shared AND the categories of third parties to whom it was sold or shared (by category of personal information and/or by category of third parties).\n3) The policy states that consumers/users can request disclosure of categories of personal information disclosed for a business purpose AND the categories of persons/recipients to whom it was disclosed.\n4) The policy states that if the business has not sold/shared (or not disclosed for business purpose), it will disclose that fact in response to a request.\n5) A clearly relevant heading (e.g., \"Right to Know What We Sell or Share\", \"Right to Know\", \"California Privacy Rights\", \"CCPA/CPRA Rights\", \"Categories of Personal Information Sold/Shared\", \"Recipients/Third Parties\") AND surrounding text grants the above request rights.\n\nImportant Notes:\n- The key requirement is a USER/CONSUMER RIGHT to request and receive these disclosures. Merely stating \"we share/sell\" or listing third parties does NOT qualify unless framed as a right to request/know.\n- \"Sold\" and \"shared\" should be interpreted as CCPA/CPRA concepts; include language about \"sharing for cross-context behavioral advertising\" if present.\n- Extract only the exact text that corresponds to the qualifying right(s). If multiple sub-rights appear, extract each relevant piece separately.\n\nNon-qualifying examples (do NOT qualify):\n- General descriptions of data sharing/sale without a consumer request/right.\n- Generic \"we may disclose\" statements without providing a right to know categories and recipients.\n- Opt-out-only language (e.g., only \"Do Not Sell or Share\") without the right-to-know disclosures.\n\nResponse Format:\nRespond using the following JSON format: [<presence_flag>, <extracted_texts>].\n- <presence_flag>: Set to '1' if the segment includes qualifying information, or '0' if it does not.\n- <extracted_texts>: A list containing the exact text of each qualifying piece of information identified (include relevant headings if they qualify). Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R17 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains information about customers' right to opt out of the processing of their personal information by the business, such as processing for purposes of the sale or sharing of personal information, targeted advertising, or profiling. The presence of a relevant heading text (e.g., \"Right to Opt out\") also qualifies. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R19": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app's privacy policy in Markdown format. Please review the segment carefully to determine whether it states that an individual (user/data subject) has the right to request the data controller (or equivalent terms such as \"we\", \"company\") and to object to, contest, or challenge a decision/result that affects them which is produced solely by automated processing/automated systems (i.e., decisions made without human involvement, including profiling).\n\nQualification Criteria (any of the following qualifies):\n1) The policy explicitly states users have the right to object to or contest decisions/results that are based solely on automated processing (\"solely automated\", \"automated decision-making\", \"automated systems\", \"without human review\"), especially where the decision produces legal or similarly significant effects.\n2) The policy explicitly provides a right to request human intervention, human review, or manual reconsideration of an automated decision that affects the user (when framed as a user right).\n3) The policy explicitly grants a right to object to profiling or automated analysis that leads to a decision/result affecting the user, when it is described as solely automated.\n4) A clearly relevant heading (e.g., \"Automated Decision-Making\", \"Profiling\", \"Your Right to Object to Automated Decisions\", \"Human Review\") AND surrounding text grants the right described above.\n\nImportant Notes:\n- The key requirement is a USER RIGHT to object/contest the result of decisions made solely by automated processing.\n- General statements that the app uses automated systems/algorithms/AI do NOT qualify unless the user is given a right to object/contest or request human review.\n- If the text describes automated decisions but explicitly says humans are involved (not solely automated), treat it as NOT qualifying unless it still grants an objection/contest right.\n\nNon-qualifying examples (do NOT qualify):\n- Descriptions of personalization, recommendations, fraud detection, or analytics without a right to object/contest or request human intervention.\n- Generic \"you may opt out of marketing\" language unless it clearly relates to solely automated decisions producing a result against the user.\n- Contact-us language without mentioning automated decision objection/contest/human review.\n\nResponse Format:\nRespond using the following JSON format: [<presence_flag>, <extracted_texts>].\n- <presence_flag>: Set to '1' if the segment includes qualifying information, or '0' if it does not.\n- <extracted_texts>: A list containing the exact text of each qualifying piece of information identified (include relevant headings if they qualify). Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"R20": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app's privacy policy in Markdown format. Please review the segment carefully to determine whether it states that an individual (user/data subject) has the right to request the data controller (or equivalent terms such as \"we\", \"company\") and to claim/seek compensation (damages) for harm arising from the unlawful/illegal processing of their personal data.\n\nQualification Criteria (any of the following qualifies):\n1) The policy explicitly states users may claim/seek compensation, damages, indemnification, or remedies for damage/harm/loss resulting from unlawful/illegal processing of their personal data.\n2) The policy explicitly mentions a right to bring a legal claim/lawsuit, seek judicial remedy, or obtain compensation specifically tied to unlawful processing or violation of data protection/privacy laws.\n3) A clearly relevant heading (e.g., \"Compensation\", \"Damages\", \"Legal Remedies\", \"Right to Compensation\", \"Judicial Remedies\") AND surrounding text grants the right to claim compensation for unlawful processing.\n\nImportant Notes:\n- The key requirement is a USER RIGHT to claim compensation for damage caused by unlawful processing.\n- General statements about \"you may complain to an authority\" do NOT qualify unless compensation/damages/legal remedy for harm is mentioned.\n- General limitation-of-liability clauses (e.g., disclaimers) do NOT qualify; they are not a user right to compensation.\n\nNon-qualifying examples (do NOT qualify):\n- Only describing complaint rights (\"contact us\" / \"complain to regulator\") without compensation/damages.\n- Only describing security incidents without stating a right to seek compensation.\n- Only stating the business may be liable or not liable, without granting a right to claim compensation.\n\nResponse Format:\nRespond using the following JSON format: [<presence_flag>, <extracted_texts>].\n- <presence_flag>: Set to '1' if the segment includes qualifying information, or '0' if it does not.\n- <extracted_texts>: A list containing the exact text of each qualifying piece of information identified (include relevant headings if they qualify). Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"E1 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. \nYour task is to review the segment carefully to determine whether it contains the information on how a consumer may appeal a controller's decision with regard to the consumer's request. \nNote: This task is distinct from determining whether the consumer has the right to lodge a complaint. Solely mentioning the right to lodge a complaint does not qualify as information on how to appeal a controller's decision.\n\nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"E24 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains a description of the process the business uses to verify a consumer request to know, delete, and correct, including any information the consumer must provide.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"E25 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains Instructions on how an authorized agent can make a request under the California Consumer Privacy Act (CCPA) on the consumer\u2019s behalf. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"E26 ": "\n\nTask Description and Instructions: \nUnder the California Consumer Privacy Act (CCPA):\n1. Businesses that have actual knowledge that it sells or shares the personal information of a consumer less than the age of 13 shall establish, document, and comply with a reasonable method for determining that the person consenting to the sale or sharing of the personal information about the child is the parent or guardian of that child. \n2. Businesses that have actual knowledge that it sells or shares the personal information of consumers at least 13 years of age and less than 16 years of age shall establish, document, and comply with a reasonable process for allowing such consumers to opt-in to the sale or sharing of their personal information. \n\nYour task is to carefully review the segment and determine whether it contains the following descriptions:\n1. Description 1: The process for determining that the person consenting to the sale or sharing of the personal information of a child under 13 is the parent or guardian of that child.\n2. Description 2: The process for allowing consumers aged 13 to less than 16 to opt-in to the sale or sharing of their personal information.\n\nResponse Format:\nPlease respond in the following JSON format: \n[ \n    <presence_flag_1>, <extracted_text_1>, \n    <presence_flag_2>, <extracted_text_2>\n]\n- presence_flag_1: Set to '1' if the segment includes Description 1, or '0' if it does not.\n- extracted_text_1: Return the exact text of Description 1 identified in the segment. If none is present, return 'None'.\n- presence_flag_2: Set to '1' if the segment includes Description 2, or '0' if it does not. \n- extracted_text_2: Return the exact text of Description 2 identified in the segment. If none is present, return 'None'.\nDo not include any additional explanations outside the JSON format to ensure ease of processing."},
# {"E27 ": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains explanations on how an opt-out preference signal will be processed for the consumer (i.e., whether the signal applies to the device, browser, consumer account, and/or offline sales, and in what circumstances) and how the consumer can use an opt-out preference signal.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant explanations, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant explanation identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"E28 ": "\n\nTask Description and Instructions: \nUnder the California Consumer Privacy Act (CCPA), if a business processes opt-out preference signals in a frictionless manner, the business shall provide information on how consumers can implement opt-out preference signals for the business to process in a frictionless manner. I will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it contains the information on how consumers can implement opt-out preference signals for the business to process in a frictionless manner.\n\nFrictionless manner:\n\"Frictionless manner\" means that the business shall not display a notification, pop-up, text, graphic, animation, sound, video, or any interstitial content in response to the opt-out preference signal. \n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes relevant information, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of each relevant piece of information identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."},
# {"O1 ": "\n\nTask Description and Instructions:\nI will provide a segment of a mobile app's privacy notice/privacy policy in Markdown format. Your job is NOT to look for a statement declaring language options. Instead, determine whether the segment itself is written in a compliant language format.\n\nCompliance Rule:\n- Output '1' if the segment is written in English OR in any Eighth Schedule language.\n- Output '0' if the segment is written in a language that is neither English nor an Eighth Schedule language, or if the segment is mixed/unclear such that compliance cannot be determined.\n\nHow to Evaluate:\n1) Detect the primary language of the segment.\n2) If the segment is multilingual, treat it as compliant ONLY if all substantial content is in English and/or Eighth Schedule language(s). If there is substantial content in other languages, output '0'.\n3) Ignore isolated proper nouns, product names, URLs, email addresses, legal citations, or short UI labels when determining the primary language.\n\nResponse Format:\nRespond using the following JSON format: [<presence_flag>, <detected_languages>].\n- <presence_flag>: '1' if compliant, otherwise '0'.\n- <detected_languages>: A list of detected language name(s) for the segment (e.g., [\"English\"], [\"Malay\"], [\"English\", \"Chinese\"], etc.). If language cannot be determined, return 'None'."},
# {"O2": "\n\nTask Description and Instructions: \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it specifies a date that the privacy policy was last updated. \n\nResponse Format: Respond using the following JSON format: [ <presence_flag>, <extracted_date> ]. \n- presence_flag: Set to '1' if the segment includes the last updated date of the privacy policy, or '0' if it does not. \n- extracted_date: The exact text of the last updated date identified in the segment. If no last updated date is found, return 'None'."},
# {"O3": "\n\nTask Description and Instructions: \nUnder the California Consumer Privacy Act (CCPA), a business that collects large amounts of personal information (the personal information of 10,000,000 or more consumers) in a calendar year shall include a metrics report in its privacy policy. This report should provide the number of privacy requests received in the previous calendar year and the average response time to those requests. \nI will provide a segment of a business's privacy policy from their website in Markdown format. Your task is to review the segment carefully to determine whether it includes the metrics report or a description of a link to that report.\n\nResponse Format: \nRespond using the following JSON format: [ <presence_flag>, <extracted_texts> ]. \n- <presence_flag>: Set to '1' if the segment includes the metrics report or a description of a link to that report, or '0' if it does not. \n- <extracted_texts>: A list containing the exact text of the metrics report or the description of a link to that report as identified in the segment. Exclude any unrelated or additional content. If none is present, return 'None'."}
# ]

In [343]:
# len(raw_items)

In [344]:
# # 2) 解析并合并（兼容 raw_items 里是 dict 或 str）
# merged = {}
# bad_rows = []

# for i, item in enumerate(raw_items, 1):
#     try:
#         # 情况1：已经是 dict（你现在就是这个情况）
#         if isinstance(item, dict):
#             obj = item

#         # 情况2：是 JSON 字符串
#         elif isinstance(item, str):
#             s = item.strip()
#             if not s or s == "]":
#                 continue
#             obj = json.loads(s)

#         else:
#             bad_rows.append((i, f"unsupported type: {type(item).__name__}"))
#             continue

#         if not isinstance(obj, dict):
#             bad_rows.append((i, "not a dict"))
#             continue

#         # 键名去首尾空格（例如 "CR2 " -> "CR2"）
#         obj = {str(k).strip(): v for k, v in obj.items()}
#         merged.update(obj)

#     except Exception as e:
#         bad_rows.append((i, str(e)))

# # 3) 保存为标准 JSON 文件
# out_path = Path("dataset4clauses/assessment_code_prompts/clause_prompts.json")
# out_path.parent.mkdir(parents=True, exist_ok=True)

# # with out_path.open("w", encoding="utf-8") as f:
# #     json.dump(merged, f, ensure_ascii=False, indent=2)

# print(f"已保存: {out_path}")
# print(f"成功条数: {len(merged)}")
# if bad_rows:
#     print("解析失败条目:")
#     for r in bad_rows[:20]:
#         print(r)

上面的json有问题，里面的每一行并不是json，所以我希望保存为：格式本质是 JSONL（每一行一个完整 JSON 对象）

In [345]:
# import json
# from pathlib import Path

# # 假设你前面已经得到 merged: {key: value}
# # 例如 merged = {"P1": "...", "P2": "...", ...}

# out_jsonl = Path("dataset4clauses/assessment_code_prompts/clause_prompts_lines.jsonl")
# out_json = Path("dataset4clauses/assessment_code_prompts/clause_prompts.json")

# # 1) 保存为每行一个完整 JSON（JSONL）
# with out_jsonl.open("w", encoding="utf-8") as f:
#     for k, v in merged.items():
#         one_row = {k: v}  # 每行一个完整json对象
#         f.write(json.dumps(one_row, ensure_ascii=False) + "\n")

# # 2) 同时保存一个标准 JSON（可选）
# with out_json.open("w", encoding="utf-8") as f:
#     json.dump(merged, f, ensure_ascii=False, indent=2)

# print("已保存 JSONL:", out_jsonl)
# print("已保存 JSON :", out_json)

如何加载这个json，然后读取里面的每一个value

In [346]:
# import json
# from pathlib import Path

In [347]:
# json_path = Path("dataset4clauses/assessment_code_prompts/clause_prompts.json")
# with json_path.open("r", encoding="utf-8") as f:
#     data = json.load(f)   # data 是 dict: {key: value}

# print("总键数:", len(data))

# # 方式1：遍历每个 value（同时拿到 key)
# for k, v in data.items():
#     print("key:", k)
#     print("value前120字符:", str(v)[:120])
#     print("-" * 60)

In [348]:
# clauses_data = []
# json_path = Path("dataset4clauses/assessment_code_prompts/clause_prompts_lines.jsonl")
# with json_path.open("r", encoding="utf-8") as f:
#     for line in f:
#         line = line.strip()
#         if line:  # 跳过空行
#             try:
#                 # 逐行解析并存入列表
#                 data = json.loads(line)
#                 clauses_data.append(data)
#             except json.JSONDecodeError as e:
#                 print(f"解析这一行时出错: {line[:50]}... 错误: {e}")
    
# # 现在 clauses_data 是一个包含所有字典的列表
# print(f"共加载了 {len(clauses_data)} 个条目"), clauses_data[0]